In [ ]:
#데이터 로드 및 탐색
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/winequality-red.csv", sep=";")

print("데이터 크기:", df.shape)
df.info()
print("결측치:")
print(df.isnull().sum())
print("기초 통계량:")
print(df.describe())
print("품질 점수 분포:", np.bincount(df["quality"]))

print("와인 품질 분포 시각화")
plt.figure(figsize=(8, 4))
plt.bar(range(len(np.bincount(df["quality"]))), np.bincount(df["quality"]))
plt.title("Wine Quality Distribution")
plt.xlabel("Quality")
plt.ylabel("Count")
plt.show()

print("알코올 분포 시각화")
plt.figure(figsize=(8, 4))
plt.hist(df['alcohol'], bins=20)
plt.title("Alcohol Distribution")
plt.xlabel("Alcohol")
plt.ylabel("Count")
plt.show()

In [ ]:
#테스트 분리
df["quality_label"] = (df["quality"] >= 7).astype(int)
X = df.drop(columns=["quality", "quality_label"])
y = df["quality_label"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

print("학습 클래스 분포:", np.bincount(y_train))
print("테스트 클래스 분포:", np.bincount(y_test))

In [ ]:
#표준화
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("표준화 후 평균:", X_train_scaled.mean(axis=0))
print("표준화 후 표준편차:", X_train_scaled.std(axis=0))

In [ ]:
#모델 학습
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

In [ ]:
#혼동행렬, 정확도, 정밀도, 재현율, f1-score로 나타낸 성능평가 및 분석
from sklearn.metrics import ( confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay)

result_df = pd.DataFrame({
    "Model": ["LogisticRegression", "DecisionTreeClassifier"],
    "Accuracy": [accuracy_score(y_test, lr_pred), accuracy_score(y_test, dt_pred)],
    "Precision": [precision_score(y_test, lr_pred), precision_score(y_test, dt_pred)],
    "Recall": [recall_score(y_test, lr_pred), recall_score(y_test, dt_pred)],
    "F1-score": [f1_score(y_test, lr_pred), f1_score(y_test, dt_pred)]
})
print("모델 성능 비교")
print(result_df.round(4))

lr_cm = confusion_matrix(y_test, lr_pred)
dt_cm = confusion_matrix(y_test, dt_pred)

print("LogisticRegression 혼동행렬:")
ConfusionMatrixDisplay(confusion_matrix=lr_cm, display_labels=["Normal", "Good"]).plot(cmap="Blues")
plt.title("LogisticRegression Confusion Matrix")
plt.show()

print("DecisionTreeClassifier 혼동행렬:")
ConfusionMatrixDisplay(confusion_matrix=dt_cm, display_labels=["Normal", "Good"]).plot(cmap="Blues")
plt.title("DecisionTreeClassifier Confusion Matrix")
plt.show()

lr_tn, lr_fp, lr_fn, lr_tp = lr_cm.ravel()
dt_tn, dt_fp, dt_fn, dt_tp = dt_cm.ravel()
print("오분류 비교:")
print(f"LogisticRegression - FP: {lr_fp}, FN: {lr_fn}")
print(f"DecisionTreeClassifier - FP: {dt_fp}, FN: {dt_fn}")

DecisionTreeClassifier는 LogisticRegression보다 정확도, 정밀도, 재현율, F1-score 모두 높은 결과를 보였다.

따라서 이번 와인 품질 분류에서는 DecisionTreeClassifier가 전반적으로 더 좋은 성능을 보였다.

다만 일부 오분류가 발생했으며, DecisionTreeClassifier의 하이퍼파라미터 튜닝을 통해 성능을 추가로 개선할 수 있다.